In [1]:
!pip install transformers torch

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [3]:
# Load Model --> BERT, xlm-Robeta
model_id  = "HPLT/hplt_bert_base_si"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model     = AutoModelForMaskedLM.from_pretrained(model_id)
model.eval()

config.json:   0%|          | 0.00/975 [00:00<?, ?B/s]

The repository HPLT/hplt_bert_base_si contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/HPLT/hplt_bert_base_si .
 You can inspect the repository content at https://hf.co/HPLT/hplt_bert_base_si.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


configuration_ltgbert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/HPLT/hplt_bert_base_si:
- configuration_ltgbert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

modeling_ltgbert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/HPLT/hplt_bert_base_si:
- modeling_ltgbert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/626M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/140 [00:00<?, ?it/s]

AttributeError: 'LtgbertForMaskedLM' object has no attribute 'all_tied_weights_keys'

In [ ]:
# Correction Function
# BERT wokrs by masking low confidence tokens and letting model predict better ones
def correct_text(text, threshold=5.0):
    inputs     = tokenizer(text, return_tensors="pt")
    input_ids  = inputs["input_ids"].clone()
    tokens     = tokenizer.convert_ids_to_tokens(input_ids[0])

    corrected_ids = input_ids.clone()

    for i in range(1, len(tokens) - 1):  # skip [CLS] and [SEP]
        masked = input_ids.clone()
        masked[0][i] = tokenizer.mask_token_id

        with torch.no_grad():
            logits = model(masked).logits

        probs       = torch.softmax(logits[0][i], dim=-1)
        top_prob, top_id = probs.max(dim=-1)

        # if BERT is confident in a different token, replace it
        original_prob = probs[input_ids[0][i]].item()
        if top_prob.item() - original_prob > threshold / 100:
            corrected_ids[0][i] = top_id

    corrected_tokens = tokenizer.convert_ids_to_tokens(corrected_ids[0])
    corrected_text   = tokenizer.convert_tokens_to_string(corrected_tokens[1:-1])
    return corrected_text

In [ ]:
# Test cell
raw_text      = "න්ක ඔන් එකෙතියෙන්නේ කපල නෑ පියෝටීවිසහ ඔයිස් දෙක මෝන්න කි තියේන සමොකද්වෙලා තියෙන්නේ වැඩකර පියෝටීවිකත් වැඩකරනා ටවඉස් එකත් හරිමේ ලයින් එකේ ඒ පහසු කන්දක මෝන් කෙසර තියෙනන ගැටලුවක් නෑ ලයින් රනාඩෙදාස් පන්සේ අසුතුනත්ත සර්ගෙවන්නතියනැච්චර ගවන්න තියෙන්නේවසසෙ පොඩ්ඩක් බලන්න කියන  මේ සමහරක් ගලට මේ කේබල් වහිනකොට ගලවනානේ අරකළුපාඩ බොක්සකින් නවා නිට් ගේලක තියෙනන හතරෙන්නේ කවුලලිව පොඩ්ඩක් හමට ගැලිල ඇති පශික්කරනකන් මේ පත්තනං ඔව කෙස හරිසහරිවකස එසටමත සවදස් සබදාසක් ස ඇමතිමකිසනා රඳඉන්න"
corrected     = correct_text(raw_text)

print("Input    :", raw_text)
print("Corrected:", corrected)